## 전처리 코드

In [1]:
import os
import json
import numpy as np
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────
ROOT_DIR_R = r"D:\VisionGuard\[라벨]bbox(실제도로환경)"
ROOT_DIR_C = r"D:\VisionGuard\[라벨]bbox(통제환경)"
SAVE_DIR   = r"D:\VisionGuard\processed"

os.makedirs(SAVE_DIR, exist_ok=True)

# ── 결과 저장용 리스트 ─────────────────────────────────
data_list    = []
normal_count = 0
drowsy_count = 0
skip_count   = 0

# ── 두 환경 폴더 순서대로 처리 ────────────────────────
for root_dir in [ROOT_DIR_R, ROOT_DIR_C]:
    json_files = list(Path(root_dir).rglob("*.json"))
    print(f"\n[{Path(root_dir).name}] JSON 파일 수: {len(json_files)}")

    for json_path in json_files:
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # ── 얼굴 감지 여부 확인 ─────────────────────
            face_vis = data["ObjectInfo"]["BoundingBox"]["Face"]["isVisible"]
            if not face_vis:
                skip_count += 1
                continue

            # ── 눈 Opened 값 추출 ───────────────────────
            leye_open = data["ObjectInfo"]["BoundingBox"]["Leye"]["isVisible"]
            reye_open = data["ObjectInfo"]["BoundingBox"]["Reye"]["isVisible"]

            # 눈이 카메라에 안 보이는 경우 제외
            if not leye_open or not reye_open:
                skip_count += 1
                continue

            leye_opened = data["ObjectInfo"]["BoundingBox"]["Leye"]["Opened"]
            reye_opened = data["ObjectInfo"]["BoundingBox"]["Reye"]["Opened"]

            # ── 핵심: Opened 값으로 정상/졸음 판단 ────────
            # 양쪽 눈 모두 열려있으면 → 정상(0)
            # 한쪽이라도 감겨있으면   → 졸음(1)
            if leye_opened == True and reye_opened == True:
                label = 0  # 정상
            else:
                label = 1  # 졸음

            # ── 눈 좌표로 EAR 계산 ─────────────────────
            leye_pos = data["ObjectInfo"]["BoundingBox"]["Leye"]["Position"]
            reye_pos = data["ObjectInfo"]["BoundingBox"]["Reye"]["Position"]

            leye_h = float(leye_pos[3]) - float(leye_pos[1])
            leye_w = float(leye_pos[2]) - float(leye_pos[0])
            reye_h = float(reye_pos[3]) - float(reye_pos[1])
            reye_w = float(reye_pos[2]) - float(reye_pos[0])

            avg_ear = ((leye_h / leye_w) + (reye_h / reye_w)) / 2 \
                      if leye_w > 0 and reye_w > 0 else 0

            # ── 결과 저장 ──────────────────────────────
            data_list.append({
                "file":         json_path.name,
                "label":        label,
                "leye_opened":  leye_opened,
                "reye_opened":  reye_opened,
                "avg_ear":      round(avg_ear, 4)
            })

            if label == 0:
                normal_count += 1
            else:
                drowsy_count += 1

        except Exception as e:
            print(f"오류: {json_path.name} - {e}")
            skip_count += 1
            continue

# ── 결과 출력 ──────────────────────────────────────────
print("\n===== 전처리 완료 =====")
print(f"정상  데이터: {normal_count}개")
print(f"졸음  데이터: {drowsy_count}개")
print(f"스킵  데이터: {skip_count}개")
print(f"전체  데이터: {normal_count + drowsy_count}개")
print(f"정상:졸음 비율 = {normal_count}:{drowsy_count}")

# ── numpy 저장 ─────────────────────────────────────────
labels = np.array([d["label"]   for d in data_list], dtype=np.int32)
ears   = np.array([d["avg_ear"] for d in data_list], dtype=np.float32)

np.save(os.path.join(SAVE_DIR, "labels.npy"), labels)
np.save(os.path.join(SAVE_DIR, "ears.npy"),   ears)

print(f"\n저장 완료: {SAVE_DIR}")
print(f"labels.npy: {labels.shape}")
print(f"ears.npy:   {ears.shape}")


[[라벨]bbox(실제도로환경)] JSON 파일 수: 186655

[[라벨]bbox(통제환경)] JSON 파일 수: 100303

===== 전처리 완료 =====
정상  데이터: 230800개
졸음  데이터: 43372개
스킵  데이터: 12786개
전체  데이터: 274172개
정상:졸음 비율 = 230800:43372

저장 완료: D:\VisionGuard\processed
labels.npy: (274172,)
ears.npy:   (274172,)


## 분류기 학습

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# ── 데이터 로드 ────────────────────────────────────────
SAVE_DIR = r"D:\VisionGuard\processed"

ears   = np.load(rf"{SAVE_DIR}\ears.npy")
labels = np.load(rf"{SAVE_DIR}\labels.npy")

print(f"ears   shape: {ears.shape}")
print(f"labels shape: {labels.shape}")
print(f"정상(0): {(labels == 0).sum()}개")
print(f"졸음(1): {(labels == 1).sum()}개")

# ── 입력 형태 변환 ─────────────────────────────────────
# sklearn은 2D 입력 필요 → (N,) → (N, 1)
X = ears.reshape(-1, 1)
y = labels

# ── 학습/검증 데이터 분리 (8:2) ───────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n학습 데이터: {len(X_train)}개")
print(f"검증 데이터: {len(X_test)}개")

# ── 모델 정의 ──────────────────────────────────────────
models = {
    "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting":   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN":                KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
}

# ── 모델별 학습 및 정확도 확인 ────────────────────────
print("\n===== 분류기 학습 결과 =====")

for name, model in models.items():
    print(f"\n[{name}] 학습 중...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"정확도: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=["정상", "졸음"]))

ears   shape: (274172,)
labels shape: (274172,)
정상(0): 230800개
졸음(1): 43372개

학습 데이터: 219337개
검증 데이터: 54835개

===== 분류기 학습 결과 =====

[RandomForest] 학습 중...
정확도: 84.01%
              precision    recall  f1-score   support

          정상       0.84      0.99      0.91     46161
          졸음       0.40      0.02      0.04      8674

    accuracy                           0.84     54835
   macro avg       0.62      0.51      0.48     54835
weighted avg       0.77      0.84      0.77     54835


[GradientBoosting] 학습 중...
정확도: 84.24%
              precision    recall  f1-score   support

          정상       0.84      1.00      0.91     46161
          졸음       0.62      0.01      0.02      8674

    accuracy                           0.84     54835
   macro avg       0.73      0.50      0.47     54835
weighted avg       0.81      0.84      0.77     54835


[LogisticRegression] 학습 중...
정확도: 84.18%
              precision    recall  f1-score   support

          정상       0.84      1.00      0.

C:\Users\blue5\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\blue5\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\blue5\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

정확도: 82.00%
              precision    recall  f1-score   support

          정상       0.84      0.96      0.90     46161
          졸음       0.22      0.05      0.08      8674

    accuracy                           0.82     54835
   macro avg       0.53      0.51      0.49     54835
weighted avg       0.74      0.82      0.77     54835

